It seems that I have (especially NuE) files missing in my iceprod production. What is going on?

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import simweights
import pickle
import os, sys
import re
import numpy as np

from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

import matplotlib as mat
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.colors as colors
import pandas as pd
import tables
import h5py
import math
from scipy.stats import mstats
import matplotlib as mpl
import matplotlib.font_manager as font_manager


In [4]:
sys.path.append("/data/user/tvaneede/GlobalFit/reco_processing/notebooks/weighting")
from weights import *
from utils import *
from selections import selection_mask
from fonts import *
from plot_utils import *

In [5]:
# Append the custom module path
sys.path.append("/data/user/tvaneede/GlobalFit/reco_processing")

simulation_datasets = {}

# Import the datasets module
from datasets import datasets as datasets
reco_versions = ["spice_tau_reco","evtgen_v4_rec_v9"]
for reco_version in reco_versions: simulation_datasets[reco_version] = getattr(datasets, reco_version)

from datasets import datasets_analysis as datasets
reco_versions = ["taureco_iceprod_v3"]
for reco_version in reco_versions: simulation_datasets[reco_version] = getattr(datasets, reco_version)

In [6]:
livetime_yr = 11.687
livetime_s  = livetime_yr * 365.25 * 24 * 3600 # 11.687 year

# weight functions
gamma_astro = 2.87
per_flavor_norm = 2.12
AstroFluxModel_HESE = create_AstroFluxModel(per_flavor_norm=per_flavor_norm, gamma_astro=gamma_astro)

In [7]:
flavor = "NuE"

In [8]:
def get_rate( hdf_file_path, nfiles ):
    hdf_file = pd.HDFStore( hdf_file_path ,'r')
    weighter = simweights.NuGenWeighter( hdf_file, nfiles)
    weights = weighter.get_weights(AstroFluxModel_HESE) * livetime_s
    rate = sum(weights)
    error = np.sqrt(np.sum(weights**2))
    return rate, error

Lets get the number of files with different methods
- ls: ls -1 path | wc -l

In [9]:
### spice
### reco_path /data/ana/Diffuse/GlobalFit_Flavor/taupede/SnowStorm/RecowithBfr/Baseline/22082
dataset_id = "22082"
subfolders = ["0000000-0000999","0001000-0001999"] 
nfiles_ls = [1000,225]

for subfolder, nfile_ls in zip(subfolders,nfiles_ls):
    hdf_path = "/data/user/tvaneede/GlobalFit/reco_processing/hdf/output/spice_tau_reco"
    hdf_file_name = f"{flavor}_{dataset_id}_{subfolder}.h5" 
    rate,error = get_rate( f"{hdf_path}/{hdf_file_name}", nfile_ls )
    print("spice_tau_reco", dataset_id, subfolder, rate, error)

spice_tau_reco 22082 0000000-0000999 54.11967042956938 0.7198691719708757
spice_tau_reco 22082 0001000-0001999 57.40525647575658 1.696750228689565


In [10]:
### evtgen_v4_rec_v9
### reco_path /data/user/tvaneede/GlobalFit/reco_processing/evtgen/output/v4/22613
dataset_id = "22613"
subfolders = ["0000000-0000999","0001000-0001999"] 
nfiles_ls = [1000,997]

for subfolder, nfile_ls in zip(subfolders,nfiles_ls):
    hdf_path = "/data/user/tvaneede/GlobalFit/reco_processing/hdf/output/evtgen_v4_rec_v9"
    hdf_file_name = f"{flavor}_{dataset_id}_{subfolder}.h5" 
    rate,error = get_rate( f"{hdf_path}/{hdf_file_name}", nfile_ls )
    print("evtgen_v4_rec_v9", dataset_id, subfolder, rate, error)

evtgen_v4_rec_v9 22613 0000000-0000999 54.05971538924764 0.724765273560978
evtgen_v4_rec_v9 22613 0001000-0001999 55.8944707324697 0.7603799464076021


In [11]:
### taureco_iceprod_v3
### reco_path /data/sim/IceCube/2023/filtered/HESE/neutrino-generator/22613/
dataset_id = "22613"
subfolders = ["0000000-0000999","0001000-0001999","0002000-0002999","0003000-0003999"] 
nfiles_ls = [996,990,987,997]
print(sum(nfiles_ls))

for subfolder, nfile_ls in zip(subfolders,nfiles_ls):
    hdf_path = "/data/user/tvaneede/GlobalFit/reco_processing/hdf/output/taureco_iceprod_v3"
    hdf_file_name = f"HESE_{flavor}_{dataset_id}_{subfolder}.h5" 
    rate,error = get_rate( f"{hdf_path}/{hdf_file_name}", nfile_ls )
    print("taureco_iceprod_v3", dataset_id, subfolder, rate, error)

    hdf_file = pd.HDFStore( f"{hdf_path}/{hdf_file_name}" ,'r')
    print("min/max", min(hdf_file["I3EventHeader"]["Run"]), max(hdf_file["I3EventHeader"]["Run"]))

# merged
hdf_path = "/data/user/tvaneede/GlobalFit/reco_processing/hdf/output/taureco_iceprod_v3/merged"
hdf_file_name = f"HESE_{flavor}_{dataset_id}.h5"
rate,error = get_rate( f"{hdf_path}/{hdf_file_name}", nfile_ls )
print("taureco_iceprod_v3", dataset_id, subfolder, rate, error)

hdf_file = pd.HDFStore( f"{hdf_path}/{hdf_file_name}" ,'r')
print("min/max", min(hdf_file["I3EventHeader"]["Run"]), max(hdf_file["I3EventHeader"]["Run"]))

3970
taureco_iceprod_v3 22613 0000000-0000999 54.088121864005444 0.726302570071921
min/max 2261300000 2261300999
taureco_iceprod_v3 22613 0001000-0001999 55.96057879815818 0.7626249493517036
min/max 2261301000 2261301999
taureco_iceprod_v3 22613 0002000-0002999 54.26972212021926 0.7409738258124152
min/max 2261302000 2261302999
taureco_iceprod_v3 22613 0003000-0003999 54.59653323627372 0.7327724459066765
min/max 2261303000 2261303999
taureco_iceprod_v3 22613 0003000-0003999 217.9234721724644 1.4747729971879977
min/max 2261300000 2261303999


Seems like there is a problem with the merging. The last subfolder is missing?! Now retrying to rerun.
Turns out: I had a scary continue in my loop... probably still for v2 of the processing.

Running v4 of the hdf, nice print statements added. Let's do a final comparison with
- number of files in folder
- number of files after checksum check
- check the checksum files
- compare with output on iceprod website


Now everything looks ok in v4. Maybe 2-20 files corrupted in the folder. 